# Chronos-2 Forecasting — DIMER `TASK-INFERENCE` tutorial

**Notebook profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0

This notebook demonstrates **zero-shot probabilistic time-series forecasting** with the repository's public `chronos2_pipeline` API. Chronos-2 supplies the pretrained forecasting model; this repository adds the DIMER-facing configuration, validation, immutable model pinning and digest checks, normalized output contract, chronological evaluation helpers, naive baselines, and provenance export.

**No training or fine-tuning occurs.** Historical observations condition the pretrained model at inference time; no model parameters or learned preprocessing state are updated. The point forecast is the model median (`q0.5`), not a statistical mean.

By the end of this notebook you will be able to:

- bootstrap the repository from its locked dependency graph and inspect the runtime;
- load the deterministic sample or a user-supplied CSV through the same DIMER validation/inference path;
- hold out the future chronologically without leaking future targets;
- resolve and verify the pinned `amazon/chronos-2` checkpoint;
- run zero-shot forecasting and interpret median/quantile outputs;
- compare tutorial forecasts with history-only baselines; and
- export forecasts, evaluation results, and immutable model/runtime provenance.

**This notebook does not demonstrate:** model training or fine-tuning, classification, anomaly detection, imputation, representation learning, calibrated prediction intervals, or production fitness.

References: [repository README](../README.md) · [model card](../MODEL_CARD.md) · [sample dataset card](../examples/sample-data/DATASET_CARD.md) · [upstream Chronos repository](https://github.com/amazon-science/chronos-forecasting) · [pinned model repository](https://huggingface.co/amazon/chronos-2)

## Prerequisites

- **Runtime:** Python 3.12; CPU is the default and supported tutorial path. CUDA is optional but not required.
- **Network:** the first model load retrieves the pinned Hugging Face snapshot unless an already verified cache entry is available.
- **Model download:** the pinned `model.safetensors` is 477,930,472 bytes (~478 MB).
- **Data:** the default sample is synthetic and bundled. BYOD expects a UTF-8 CSV with unique headers and `series_id`, `timestamp`, and `target` columns.
- **Data handling:** inference runs locally in the notebook runtime. A BYOD CSV is read into that runtime and is not sent by this notebook to an external inference service. Hugging Face network access is used only to acquire/confirm the pinned model snapshot.
- **Sensitive data:** do not upload confidential, restricted, personal, or otherwise sensitive data to a hosted notebook environment unless you are authorized to place it there.

The default sample path is non-interactive. Upload dialogs and the optional covariate demonstration are disabled unless explicitly enabled.


## 1. Bootstrap the repository and locked runtime

This cell ensures the notebook is attached to a repository checkout, then installs the exact exported dependency graph from `requirements.lock.txt` with hash verification and installs the local package with dependency resolution disabled. It uses `uv` when available and otherwise falls back to the runtime's `pip`.

Set `DIMER_TUTORIAL_REF` only when verifying a specific branch/commit in automation. The normal public path uses `main`.

If installation replaces a core package that was already imported in the current kernel, the cell fails before model use with an explicit restart instruction. In Colab, restart the session and rerun from the top; this avoids mixing old in-memory modules with newly installed distributions.


In [ ]:
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline.git"
REPO_NAME = "chronos-2-forecasting-pipeline"
REPO_REF = os.environ.get("DIMER_TUTORIAL_REF", "main")
SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(["git", "clone", "--filter=blob:none", "-q", REPO_URL, str(checkout)], check=True)
    if REPO_REF != "main":
        subprocess.run(
            ["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", REPO_REF],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(checkout), "checkout", "-q", "main"], check=True)
        subprocess.run(["git", "-C", str(checkout), "pull", "--ff-only", "-q", "origin", "main"], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked_modules = {"numpy": "numpy", "pandas": "pandas", "torch": "torch"}
    loaded_before = {
        module: getattr(sys.modules[module], "__version__", None)
        for module in tracked_modules.values()
        if module in sys.modules
    }
    if shutil.which("uv"):
        subprocess.run(
            [
                "uv",
                "pip",
                "install",
                "--python",
                sys.executable,
                "--require-hashes",
                "-r",
                str(ROOT / "requirements.lock.txt"),
            ],
            check=True,
        )
        subprocess.run(
            [
                "uv",
                "pip",
                "install",
                "--python",
                sys.executable,
                "--no-deps",
                "-e",
                str(ROOT),
            ],
            check=True,
        )
    else:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--require-hashes",
                "-r",
                str(ROOT / "requirements.lock.txt"),
            ],
            check=True,
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)],
            check=True,
        )
    stale = []
    for distribution, module in tracked_modules.items():
        if module in loaded_before:
            installed = importlib.metadata.version(distribution)
            loaded = loaded_before[module]
            if loaded is not None and loaded != installed:
                stale.append(f"{module}: loaded={loaded}, installed={installed}")
    if stale:
        raise RuntimeError(
            "Core dependencies changed while older modules were already loaded: "
            + "; ".join(stale)
            + ". Restart the notebook session/runtime, then rerun from the top."
        )

print("repository:", ROOT)
print("repository ref:", REPO_REF)


## 2. Report runtime identity and operational limits

Successful output here establishes which interpreter and principal model libraries the tutorial will actually use. It also surfaces DIMER request ceilings **before forecast execution**.

Chronos-2's pinned checkpoint supports an 8,192-step model context and a native 1,024-step forecast horizon. DIMER additionally caps a single request at 1,000 series IDs, 64 targets, 64 covariates, 5,000,000 rows, 8,192 context steps, and 4,096 forecast steps. Horizons beyond the model-native 1,024 steps require explicit autoregressive unrolling in the pipeline.

The default tutorial uses CPU, float32-compatible execution, and a 12-step horizon. It uses deterministic synthetic data and a deterministic chronological split; no random split, stochastic sampling, or training initialization occurs. Minor floating-point differences can still arise across framework/hardware builds, and latency is inherently run-dependent.


In [ ]:
import platform

from chronos2_pipeline import ResourceLimits, runtime_versions

limits = ResourceLimits()
versions = runtime_versions()

print("Python:", platform.python_version())
print("chronos-forecasting:", versions.get("chronos-forecasting"))
print("torch:", versions.get("torch"))
print("transformers:", versions.get("transformers"))
print("device required by default path: CPU")
print("DIMER request limits:", limits)
print("Pinned-model context limit: 8192")
print("Pinned-model native prediction length: 1024")


## 3. Load the bundled sample or BYOD

The default sample is **synthetic, deterministic teaching data**, not benchmark evidence. Its generator, provenance, license, and digest are documented in `examples/sample-data/DATASET_CARD.md`. The notebook verifies the checked-in sample against `SHA256SUMS` before reading it.

To use your own data in Colab, set `USE_BYOD = True`. Expected schema:

| Column | Meaning | Requirement |
|---|---|---|
| `series_id` | series/entity identifier | non-null |
| `timestamp` | observation time | parseable, strictly regular and contiguous per series |
| `target` | value to forecast | finite numeric |

Additional numeric covariates are permitted by the pipeline. CSV headers must be unique; duplicates are rejected **before** pandas can rename them. The production-facing pipeline then performs the remaining schema, regularity, horizon, and resource-limit validation before model execution.

Automation can set `DIMER_BYOD_PATH` to exercise the same CSV-ingestion branch without an upload dialog.


In [ ]:
import csv
import hashlib
import io
import json
from collections import Counter

import pandas as pd

from chronos2_pipeline import (
    ForecastConfig,
    chronological_holdout,
    evaluate_forecast,
    forecast,
    last_value_baseline,
    load_pinned_model,
    seasonal_naive_baseline,
)

USE_BYOD = False  # @param {type:"boolean"}
PREDICTION_LENGTH = 12  # @param {type:"integer"}
BYOD_PATH = os.environ.get("DIMER_BYOD_PATH")

def reject_duplicate_csv_headers(payload: bytes) -> None:
    text = payload.decode("utf-8-sig")
    reader = csv.reader(io.StringIO(text))
    try:
        header = next(reader)
    except StopIteration as exc:
        raise ValueError("CSV is empty; expected a header row.") from exc
    duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
    if duplicates:
        raise ValueError(
            "CSV contains duplicate column name(s) that would be ambiguous: "
            + ", ".join(repr(name) for name in duplicates)
        )

def read_checked_csv(payload: bytes) -> pd.DataFrame:
    reject_duplicate_csv_headers(payload)
    return pd.read_csv(io.BytesIO(payload))

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def expected_digest(manifest: Path, filename: str) -> str:
    records = {}
    for line in manifest.read_text(encoding="utf-8").splitlines():
        digest, name = line.split(maxsplit=1)
        records[name.strip()] = digest
    if filename not in records:
        raise ValueError(f"{filename!r} is not recorded in {manifest}.")
    return records[filename]

if BYOD_PATH:
    payload = Path(BYOD_PATH).read_bytes()
    input_source = f"BYOD path: {BYOD_PATH}"
    frame = read_checked_csv(payload)
elif USE_BYOD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Interactive BYOD upload is available in Colab. "
            "Outside Colab, set DIMER_BYOD_PATH to a CSV file."
        ) from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV for the BYOD path.")
    name, payload = next(iter(uploaded.items()))
    input_source = f"BYOD upload: {name}"
    frame = read_checked_csv(payload)
else:
    sample_path = ROOT / "examples" / "sample-data" / "chronos_univariate.csv"
    payload = sample_path.read_bytes()
    digest = sha256_bytes(payload)
    expected = expected_digest(sample_path.parent / "SHA256SUMS", sample_path.name)
    if digest != expected:
        raise ValueError(
            f"Bundled sample digest mismatch: expected {expected}, observed {digest}."
        )
    input_source = f"bundled synthetic sample: {sample_path}"
    frame = read_checked_csv(payload)

input_sha256 = sha256_bytes(payload)
print(input_source)
print("input sha256:", input_sha256)
print(frame.head())
print(f"rows={len(frame)}, columns={list(frame.columns)}")


## 4. Hold out the future chronologically

This tutorial uses a **single chronological tail holdout**: the last `PREDICTION_LENGTH` timestamps from each series are removed and retained as ground truth. This preserves temporal order and prevents future target values from entering model context. It is not a random row split.

For genuine future forecasting there is no held-out truth; forecast from all observations available up to the forecast origin and evaluate later when outcomes arrive.


In [ ]:
config = ForecastConfig(
    target="target",
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=[0.1, 0.5, 0.9],
    device="cpu",
)

split = chronological_holdout(frame, config)
for series_id in split.history[config.id_column].drop_duplicates():
    history_block = split.history[split.history[config.id_column] == series_id]
    truth_block = split.truth[split.truth[config.id_column] == series_id]
    assert history_block[config.timestamp_column].max() < truth_block[config.timestamp_column].min()

print("holdout method: final", split.horizon, "timestamps per series")
print("history rows:", len(split.history), "| held-out rows:", len(split.truth))
print("context through:", split.history[config.timestamp_column].max())
print("held-out future starts:", split.truth[config.timestamp_column].min())


## 5. Resolve and verify the pinned Chronos-2 checkpoint

`load_pinned_model()` accepts only the repository-approved `amazon/chronos-2` revision. Before loading model state it verifies the resolved snapshot, `model.safetensors` byte size/SHA-256, and `config.json` SHA-256, and refuses pickle-style weight files. No model-repository remote code is requested by this DIMER path.

Successful output below identifies the effective immutable model revision and the device/dtype actually used.


In [ ]:
model = load_pinned_model(device="cpu")

print("model:", model.identity.model_id)
print("revision:", model.identity.revision)
print("weights sha256:", model.identity.weights_sha256)
print("config sha256:", model.identity.config_sha256)
print("model context length:", model.identity.model_context_length)
print("model native prediction length:", model.identity.model_prediction_length)
print("device/dtype:", model.device, model.dtype)


## 6. Run zero-shot forecasting

The repository's public `forecast()` function validates the request, calls the pinned upstream model, verifies the upstream output layout, and normalizes the result.

Output contract:

- `series_id` — input series identifier;
- `timestamp` — forecast timestamp;
- `target_name` — target column being forecast;
- `prediction` — point forecast, defined here as the model median (`q0.5`);
- `q0.1`, `q0.5`, `q0.9` — requested model quantiles.

Model quantiles summarize the model's predictive distribution. They are **not guaranteed frequentist confidence intervals** and are not asserted to have calibrated coverage on a new domain.


In [ ]:
result = forecast(split.history, config, model)

print(result.forecast.head())
print("output columns:", list(result.forecast.columns))
print("latency_seconds:", result.inference["latency_seconds"])
print("effective_context_length:", result.inference["effective_context_length"])
print("effective_prediction_length:", result.inference["effective_prediction_length"])


## 7. Evaluate the held-out future and compare history-only baselines

All methods are scored on the **same chronological held-out timestamps**.

- **MAE** is mean absolute error in the target's units; lower is better and it gives each absolute miss linear weight.
- **RMSE** is the square root of mean squared error in target units; lower is better and larger misses receive more weight.
- **Pinball loss** evaluates an individual forecast quantile asymmetrically; lower is better.
- **Empirical interval coverage** is the fraction of held-out truths falling between the requested outer quantiles. It describes this tutorial holdout only and is not a future-domain coverage guarantee.

The last-value baseline repeats the most recent historical value. The seasonal-naive baseline repeats the value from one seasonal period earlier and is shown only when the observed data are hourly, where a 24-step daily season is meaningful for this example.

Metrics from the bundled synthetic sample are **tutorial/sanity evidence**, not benchmark measurements. Aggregate MAE/RMSE pool rows and can be scale-dominated on heterogeneous multi-series or multi-target BYOD; inspect per-series results for such data.


In [ ]:
evaluation = evaluate_forecast(result.forecast, split.truth, config)

last_value = last_value_baseline(split.history, split.truth, config)
last_value_evaluation = evaluate_forecast(last_value, split.truth, config)

sorted_history = split.history.sort_values([config.id_column, config.timestamp_column])
steps = (
    sorted_history.groupby(config.id_column)[config.timestamp_column]
    .diff()
    .dropna()
)
is_hourly = not steps.empty and (steps == pd.Timedelta(hours=1)).all()

seasonal_evaluation = None
if is_hourly:
    seasonal = seasonal_naive_baseline(
        split.history,
        split.truth,
        config,
        season_length=24,
    )
    seasonal_evaluation = evaluate_forecast(seasonal, split.truth, config)

print("Tutorial metric estimation: single chronological tail holdout")
print("Chronos-2:", evaluation.aggregate)
print("Last-value:", last_value_evaluation.aggregate)
if seasonal_evaluation is not None:
    print("Seasonal-naive (24 hourly steps):", seasonal_evaluation.aggregate)
else:
    print("Seasonal-naive skipped: uploaded data are not uniformly hourly.")
print("Quantile evaluation:")
print(evaluation.quantiles)
print("Per-series evaluation:")
print(evaluation.per_series)


## 8. Visualize one series and target

For multi-series or multi-target BYOD, this compact visualization deliberately selects one `(series_id, target_name)` pair instead of interleaving unrelated lines. The CSV exports in the next section remain authoritative for all rows.


In [ ]:
from html import escape

def write_line_svg(path, layers, *, title, width=760, height=280):
    all_values = [float(value) for _, values in layers for value in values]
    if not all_values:
        raise ValueError("cannot plot an empty series")
    low, high = min(all_values), max(all_values)
    span = high - low or 1.0
    max_points = max(len(values) for _, values in layers)
    left, right, top, bottom = 48, width - 20, 30, height - 38

    def point(index, value):
        x = left + (right - left) * index / max(max_points - 1, 1)
        y = bottom - (bottom - top) * (float(value) - low) / span
        return f"{x:.1f},{y:.1f}"

    strokes = ["#111827", "#2563eb", "#dc2626", "#059669"]
    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
        f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{escape(title)}</text>',
        f'<line x1="{left}" y1="{bottom}" x2="{right}" y2="{bottom}" stroke="#9ca3af"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#9ca3af"/>',
    ]
    for idx, (label, values) in enumerate(layers):
        points = " ".join(point(i, value) for i, value in enumerate(values))
        stroke = strokes[idx % len(strokes)]
        svg.append(f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points}"/>')
        svg.append(
            f'<text x="{left + 120 * idx}" y="{height - 10}" '
            f'font-family="sans-serif" font-size="12" fill="{stroke}">{escape(str(label))}</text>'
        )
    svg.append("</svg>")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(svg), encoding="utf-8")
    return path

series_to_plot = result.forecast["series_id"].iloc[0]
target_to_plot = result.forecast["target_name"].iloc[0]

forecast_frame = result.forecast[
    (result.forecast["series_id"] == series_to_plot)
    & (result.forecast["target_name"] == target_to_plot)
].sort_values("timestamp")
truth_frame = split.truth[
    split.truth[config.id_column] == series_to_plot
].sort_values(config.timestamp_column)

OUTPUT_DIR = Path(os.environ.get("DIMER_OUTPUT_DIR", ROOT / "outputs"))
forecast_svg = write_line_svg(
    OUTPUT_DIR / "chronos_forecast.svg",
    [
        ("q0.1", forecast_frame["q0.1"].tolist()),
        ("median", forecast_frame["prediction"].tolist()),
        ("q0.9", forecast_frame["q0.9"].tolist()),
        ("truth", truth_frame[target_to_plot].tolist()),
    ],
    title=f"{series_to_plot} / {target_to_plot}: held-out future",
)

try:
    from IPython.display import SVG, display
    display(SVG(filename=str(forecast_svg)))
except ImportError:
    print("Forecast SVG written to", forecast_svg)


## 9. Export forecasts, metrics, and provenance

The forecast CSV is the normalized machine-readable DIMER result. Evaluation is exported both as aggregate/quantile JSON and per-series CSV. Provenance records the immutable model identity/revision, runtime and inference configuration, plus this notebook profile and the SHA-256 identity of the input bytes.

These files contain no credentials. A BYOD input digest is an identifier of the uploaded bytes; treat it as metadata derived from your data and retain/disclose it according to your own data-governance rules.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

forecast_path = OUTPUT_DIR / "chronos_forecast.csv"
provenance_path = OUTPUT_DIR / "chronos_provenance.json"
metrics_path = OUTPUT_DIR / "chronos_evaluation.json"
per_series_path = OUTPUT_DIR / "chronos_evaluation_per_series.csv"

result.forecast.to_csv(forecast_path, index=False)
evaluation.per_series.to_csv(per_series_path, index=False)

tutorial_provenance = {
    **result.provenance,
    "tutorial": {
        "notebook_profile": "TASK-INFERENCE",
        "notebook_spec": "1.0",
        "input_source": input_source,
        "input_sha256": input_sha256,
        "evaluation": "single chronological tail holdout",
    },
}
provenance_path.write_text(
    json.dumps(tutorial_provenance, indent=2, default=str),
    encoding="utf-8",
)

metric_payload = {
    "evidence_scope": "tutorial/sanity metrics; not benchmark or production-fitness evidence",
    "estimation_procedure": "single chronological tail holdout",
    "chronos2": evaluation.aggregate,
    "last_value": last_value_evaluation.aggregate,
    "seasonal_naive_24": (
        seasonal_evaluation.aggregate if seasonal_evaluation is not None else None
    ),
    "quantiles": evaluation.quantiles.to_dict("records"),
}
metrics_path.write_text(
    json.dumps(metric_payload, indent=2, default=str),
    encoding="utf-8",
)

for path in (forecast_path, provenance_path, metrics_path, per_series_path):
    print("wrote:", path)


## Optional: known-future covariates (Mode D)

Chronos-2 can use numeric covariates when the future covariate values are genuinely known at forecast time. This demonstration is disabled by default.

The deterministic generator creates historical `demand`, `temperature`, and `holiday`, while the future table contains **only** `temperature` and `holiday`—never future `demand`. The notebook constructs these frames in memory, so enabling this section does not rewrite tracked repository files.

Automation may set `DIMER_RUN_COVARIATE_DEMO=1` to execute this branch.


In [ ]:
RUN_COVARIATE_DEMO = False  # @param {type:"boolean"}
if os.environ.get("DIMER_RUN_COVARIATE_DEMO") == "1":
    RUN_COVARIATE_DEMO = True

if RUN_COVARIATE_DEMO:
    import importlib.util

    generator_path = ROOT / "examples" / "sample-data" / "generate_samples.py"
    module_spec = importlib.util.spec_from_file_location(
        "chronos_sample_generator",
        generator_path,
    )
    if module_spec is None or module_spec.loader is None:
        raise RuntimeError(f"Could not load sample generator from {generator_path}.")
    generator = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(generator)
    samples = generator.build_samples()

    cov_history = samples["chronos_covariates_history.csv"]
    cov_future = samples["chronos_covariates_future.csv"]
    if "demand" in cov_future.columns:
        raise ValueError("Future covariate table must not contain the target 'demand'.")

    cov_config = ForecastConfig(
        target="demand",
        prediction_length=24,
        quantile_levels=[0.1, 0.5, 0.9],
        device="cpu",
    )
    cov_result = forecast(cov_history, cov_config, model, cov_future)
    print(cov_result.forecast.head())
    print(
        "known-future covariates:",
        cov_result.inference["known_future_covariate_names"],
    )
else:
    print("Covariate demo skipped. Set DIMER_RUN_COVARIATE_DEMO=1 or enable the parameter to run it.")


## Interpretation, limits, and next steps

A successful default run **proves** that, in the recorded runtime, the repository can install from its locked dependency graph, verify and load the pinned Chronos-2 model, preserve a chronological forecast/evaluation boundary, execute the public DIMER forecast API on the tutorial sample, compute the documented tutorial metrics/baselines, and write machine-readable forecasts plus provenance.

It **does not prove** that Chronos-2 is accurate, calibrated, robust, fair, safe, or production-ready for your domain. The bundled sample is intentionally simple synthetic teaching data; its metrics are not benchmark evidence. Quantiles are model quantiles rather than guaranteed confidence intervals. BYOD results require domain-appropriate backtesting, leakage controls, baseline selection, calibration checks, and operational review.

Current contract limits include fixed-width regular frequencies; monthly, quarterly, yearly, business-day, irregular, and gappy calendars remain outside the supported path. Forecasting is the supported capability here; classification, anomaly detection, imputation, embeddings, model training, and fine-tuning are not demonstrated.

Useful next experiments are to evaluate a real chronological backtest over multiple forecast origins, compare domain-relevant seasonal baselines, test multiple series with scale-aware metrics, and assess empirical quantile coverage on representative historical data before operational use.
